# Nuclear Isotope Project - Phase 1 country analysis
**Before you start (one time):** open the shared `phase1_outputs` folder in Google Drive -> right-click -> *Organize* -> *Add shortcut* -> My Drive. Then run this notebook top to bottom.

**You only edit Cell 3** (your country). Outputs are saved to your country's folder in `phase1_outputs/`: `profile.png`, `peers_<metric>.png`, `global_context.png`, `narrative_<Country>.md`.

## Cell 1 - Mount Drive and find the shared data

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive')
except ImportError:                      # running outside Colab
    ROOT = Path(os.environ.get('LOCAL_ROOT', '.'))

candidates = [ROOT / 'phase1_outputs', ROOT / 'nuclear_isotope_project' / 'phase1_outputs']
DATA = next((p for p in candidates if (p / 'phase1_country_baseline.csv').exists()), None)
if DATA is None:
    raise SystemExit('Cannot find phase1_outputs. Add the shared folder as a shortcut in My Drive (see top of notebook).')
print('Using data folder:', DATA)

## Cell 2 - Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

## Cell 3 - Your country (the only cell you edit)

In [ ]:
MY_COUNTRY = 'China'        # 'China' / 'United States of America' /
                            # 'Korea, Republic of' / 'Russian Federation' / 'France'
PEER_METRIC = 'Avg_age_yrs' # metric for the peer comparison chart. Options:
                            # 'Number of reactors', 'Capacity_GW', 'Lifetime_TWh',
                            # 'Avg_age_yrs', 'Productivity_TWh_per_GW'

## Cell 4 - Load and validate

In [ ]:
FOLDERS = {'China': 'china_analysis', 'United States of America': 'usa_analysis',
           'Korea, Republic of': 'korea_analysis', 'Russian Federation': 'russia_analysis',
           'France': 'france_analysis'}

baseline = pd.read_csv(DATA / 'phase1_country_baseline.csv')
fleet_ts = pd.read_csv(DATA / 'fleet_ts.csv', index_col='Year')

if MY_COUNTRY not in baseline['Country'].tolist():
    raise SystemExit(f"Country '{MY_COUNTRY}' not found. Valid: {baseline['Country'].tolist()}")
if PEER_METRIC not in baseline.columns:
    raise SystemExit(f"Metric '{PEER_METRIC}' not found. Valid: {baseline.columns.tolist()[1:]}")

# Output folder: the shared one if you have edit access, otherwise your own Drive
OUT = DATA / FOLDERS[MY_COUNTRY]
try:
    OUT.mkdir(exist_ok=True); (OUT / '.write_test').write_text('ok'); (OUT / '.write_test').unlink()
except OSError:
    OUT = ROOT / 'nuclear_isotope_analysis' / FOLDERS[MY_COUNTRY]
    OUT.mkdir(parents=True, exist_ok=True)
    print('No write access to the shared folder - saving to your own Drive instead.')

r = baseline.loc[baseline['Country'] == MY_COUNTRY].iloc[0]
print(f'Loaded {MY_COUNTRY}  ->  saving to {OUT}')
print(r.to_string())

## Cell 5 - Fleet profile (country vs the other four)

In [ ]:
def plot_fleet_profile(save=True):
    items = [('Number of reactors', 'Reactors'), ('Capacity_GW', 'Capacity (GW)'),
             ('Avg_age_yrs', 'Mean reactor age (years)')]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (col, label) in zip(axes, items):
        d = baseline.sort_values(col, ascending=False)
        colours = ['#d95f02' if c == MY_COUNTRY else '#b8c4cc' for c in d['Country']]
        ax.bar(range(len(d)), d[col], color=colours)
        ax.set_xticks(range(len(d)))
        ax.set_xticklabels([c.split(',')[0].replace('United States of America', 'USA')
                             .replace('Russian Federation', 'Russia') for c in d['Country']],
                            rotation=25, ha='right')
        ax.set_title(label)
    fig.suptitle(f'Fleet profile - {MY_COUNTRY} (highlighted) vs peers')
    fig.tight_layout()
    if save:
        fig.savefig(OUT / 'profile.png', dpi=150)
    plt.show()
    return fig

plot_fleet_profile();

## Cell 6 - Peer comparison on one metric

In [ ]:
def plot_peers(metric=PEER_METRIC, save=True):
    d = baseline.sort_values(metric)
    colours = ['#d95f02' if c == MY_COUNTRY else '#b8c4cc' for c in d['Country']]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(d['Country'], d[metric], color=colours)
    for y, v in enumerate(d[metric]):
        ax.text(v, y, f' {v:,.1f}', va='center', fontsize=9)
    ax.set_xlabel(metric); ax.set_title(f'{metric} - {MY_COUNTRY} vs peers')
    ax.margins(x=0.15)
    fig.tight_layout()
    if save:
        fig.savefig(OUT / f'peers_{metric}.png', dpi=150)
    plt.show()
    return fig

plot_peers();

## Cell 7 - Global context (world fleet over time)

In [ ]:
def plot_global_context(save=True):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(fleet_ts.index, fleet_ts['Operational_GW'], color='#1b7f79', lw=2, label='Operational')
    ax.plot(fleet_ts.index, fleet_ts['Suspended_GW'], color='#d95f02', lw=2, label='Suspended')
    ax.axvline(2011, color='grey', ls='--', lw=1); ax.text(2012, 120, 'Fukushima (2011)', fontsize=9, color='grey')
    share = r['Capacity_GW'] / fleet_ts['Operational_GW'].iloc[-1] * 100
    ax.set_title(f'World nuclear fleet - {MY_COUNTRY} = {r["Capacity_GW"]:.1f} GW ({share:.0f}% of {fleet_ts.index[-1]} operational)')
    ax.set_xlabel('Year'); ax.set_ylabel('Net electrical capacity (GW)'); ax.legend()
    fig.tight_layout()
    if save:
        fig.savefig(OUT / 'global_context.png', dpi=150)
    plt.show()
    return fig

plot_global_context();

## Cell 8 - Write your narrative
Edit the text in `narrative` below (2-4 sentences per point), then run. The key-figures table fills itself in.

In [ ]:
narrative = '''
- **Fleet size and scale:** (write here)
- **Age and renewal:** (write here)
- **Productivity vs peers:** (write here)
- **Context (global fleet / Fukushima):** (write here)
'''

labels = {'Number of reactors': 'Reactors', 'Capacity_GW': 'Capacity (GW)', 'Lifetime_TWh': 'Lifetime generation (TWh)',
          'Avg_age_yrs': 'Mean age (yrs)', 'Productivity_TWh_per_GW': 'Productivity (TWh per GW)',
          'First_construction_year': 'First construction year', 'First_grid_year': 'First grid year',
          'First_unit_lead_time_yrs': 'First-unit lead time (yrs)'}
rows = '\n'.join(f'| {labels[k]} | {r[k]:,.2f} |' if k != 'Number of reactors' and not k.endswith('year')
                  else f'| {labels[k]} | {int(r[k])} |' for k in labels)
narrative_md = f'''# {MY_COUNTRY} - Phase 1 country report

## Key figures
| Metric | Value |
|---|---|
{rows}

## Narrative
{narrative}
'''
(OUT / f'narrative_{MY_COUNTRY.replace(" ", "_")}.md').write_text(narrative_md)
print('Saved narrative to', OUT)

## Cell 9 - Check your saved files

In [ ]:
print(f'Your outputs in {OUT}:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:35} {p.stat().st_size/1024:7.1f} KB')